# Surgical Phase Recognition — Dataset Setup & EDA (Colab)

This notebook sets up a Colab environment, pulls CholecT50 resources, optionally downloads the public challenge validation set, and runs basic dataset examination.

In [ ]:
!nvidia-smi || true
!python -V

In [ ]:
# Core libs
!pip -q install -U pandas numpy matplotlib seaborn opencv-python tqdm

In [ ]:
from pathlib import Path
import os, zipfile, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('/content/surgical-phase-colab')
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
print('Working dir:', ROOT)

## 1) Pull helper repo + docs

In [ ]:
if not Path('cholect50').exists():
    !git clone --depth 1 https://github.com/CAMMA-public/cholect50.git
else:
    print('cholect50 repo already exists')

!ls -la cholect50
!ls -la cholect50/docs

## 2) (Optional) Download public Challenge Validation set

The full CholecT50 release requires request access. This public challenge split is enough for pipeline testing and EDA demos.

In [ ]:
import urllib.request
url = 'https://s3.unistra.fr/camma_public/datasets/cholect50/CholecT50-Challenge-Validation.zip'
zip_path = ROOT / 'CholecT50-Challenge-Validation.zip'
data_dir = ROOT / 'data'

if not zip_path.exists():
    print('Downloading challenge validation zip...')
    urllib.request.urlretrieve(url, zip_path)
else:
    print('Zip already exists:', zip_path)

data_dir.mkdir(exist_ok=True)
if not any(data_dir.iterdir()):
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(data_dir)
    print('Extracted to', data_dir)
else:
    print('Data dir already populated:', data_dir)

## 3) Basic file-level examination

In [ ]:
all_files = list(data_dir.rglob('*'))
num_files = sum(p.is_file() for p in all_files)
num_dirs = sum(p.is_dir() for p in all_files)
print(f'Files: {num_files} | Dirs: {num_dirs}')

# Print top-level tree
for p in sorted(data_dir.iterdir()):
    print('-', p.name)

## 4) Locate annotation tables and inspect columns

In [ ]:
ann_candidates = [p for p in data_dir.rglob('*') if p.suffix.lower() in ['.csv', '.txt', '.json']]
print('Annotation-like files found:', len(ann_candidates))
for p in ann_candidates[:30]:
    print(p)

# Try reading CSV/TXT annotation files
tables = {}
for p in ann_candidates:
    if p.suffix.lower() in ['.csv', '.txt']:
        try:
            df = pd.read_csv(p)
            if len(df.columns) > 1:
                tables[str(p)] = df
        except Exception:
            pass

print('Readable tabular files:', len(tables))
for k, df in list(tables.items())[:10]:
    print('\n', k)
    print('shape:', df.shape)
    print('columns:', list(df.columns)[:20])
    display(df.head(2))

## 5) Quick label distribution sketch (if phase labels present)

In [ ]:
phase_col = None
phase_df = None
for path, df in tables.items():
    for c in df.columns:
        cl = str(c).lower()
        if 'phase' in cl:
            phase_col = c
            phase_df = df
            print('Using phase column', c, 'from', path)
            break
    if phase_col is not None:
        break

if phase_col is not None:
    vc = phase_df[phase_col].value_counts().sort_values(ascending=False)
    plt.figure(figsize=(8,4))
    vc.plot(kind='bar')
    plt.title('Phase label distribution')
    plt.tight_layout()
    plt.show()
else:
    print('No explicit phase column found automatically. Inspect tables above manually.')

## 6) Show random sample frames (if images exist)

In [ ]:
import random, cv2
img_files = [p for p in data_dir.rglob('*') if p.suffix.lower() in ['.jpg', '.jpeg', '.png']]
print('Image files found:', len(img_files))

if img_files:
    picks = random.sample(img_files, min(6, len(img_files)))
    plt.figure(figsize=(12,8))
    for i,p in enumerate(picks,1):
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        plt.subplot(2,3,i)
        plt.imshow(img)
        plt.title(p.name[:30])
        plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No images found in extracted data path.')

## 7) Next steps

- Wire this EDA into a dataloader for phase recognition
- Build baseline training notebook
- Add metric reporting: accuracy, macro-F1, confusion matrix
- Add error analysis by phase transitions